In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_testing_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
# run_pattern = "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
run_pattern = "chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/test/model_epoch030/test_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_0105_064335', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_0105_075306', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_0105_090239', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_0105_101212', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_0105_112151', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_0105_123134', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_0105_134115', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_0105_145050']


In [4]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy(deep=True)
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'camels_01411300': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.53 1.32 1.29 ... nan nan
       streamflow_sim  (date, time_step) float32 15kB 1.564 1.422 ... 0.2889 0.2912}},
 'camels_01466500': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 0.56 0.56 0.56 ... nan nan
       streamflow_sim  (date, time_step) float32 15kB 0.6764 0.6744 ... 0.5302}},
 'camels_01487000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
   

In [5]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}

for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    obs = xr_ds['streamflow_obs']
    sim = xr_ds['streamflow_sim']
    
    # Skip basin if all obs or sim are NaN
    if obs.isnull().all() or sim.isnull().all():
        print(f"Skipping {basin_id} — all observed/simulated values are NaN")
        continue
    
    all_metrics[basin_id] = calculate_metrics(
        obs=obs,
        sim=sim,
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'
df_metrics

,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
camels_01411300,0.729151,0.422859,0.650276,0.863048,1.036429,0.870323,0.975244,-0.026852,1.148135,-3.682321,15.097791,0.500000,0.120000,26.176188
camels_01466500,0.558276,0.169622,0.411852,0.773942,1.122271,0.809864,0.999748,-0.000382,15.530949,-6.273243,-0.127361,0.333333,0.428571,31.001966
camels_01487000,0.633731,0.660727,0.812851,0.801432,0.960514,0.811223,1.047262,0.045449,-9.817204,-1.257040,-56.537991,0.200000,0.428571,47.075790
camels_01638480,0.702829,2.656059,1.629742,0.662827,0.761079,0.846208,0.818476,-0.077406,-28.990427,-19.522356,-410.021454,0.272727,0.250000,47.908890
camels_01644000,0.675629,1.342643,1.158725,0.807277,0.920249,0.827347,1.031190,0.015595,-7.258182,-14.767265,13.077731,0.300000,0.321429,31.916830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
camels_09505800,0.295426,0.072883,0.269969,0.642822,0.995090,0.646199,0.951246,-0.020592,-7.840632,176.749161,-485.933441,1.000000,0.640000,69.611481
camels_10259000,-1.220223,0.347417,0.589421,-0.266042,2.254542,0.857183,1.092686,0.048630,61.317993,-13.497834,79.869217,0.500000,0.500000,111.200020
camels_11451100,0.833796,1.841203,1.356909,0.861464,1.062391,0.924211,1.097752,0.035744,3.098931,-17.400896,73.663864,0.571429,0.263158,37.347656


In [6]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(ensemble_metrics_dir/f"{save_name}.csv")

In [7]:
df_metrics.median()

NSE              0.756569
MSE              0.795048
RMSE             0.891655
KGE              0.746728
Alpha-NSE        0.908173
Pearson-r        0.892143
Beta-KGE         1.062381
Beta-NSE         0.026000
FHV             -8.212822
FMS            -23.062473
FLV              5.033632
Peak-Timing      0.333333
Missed-Peaks     0.333333
Peak-MAPE       41.640968
dtype: float64